In [1]:
!pip install openpyxl


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\RESHAM\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import os
print(os.getcwd())
data = pd.read_excel("dataset/PCOS_data_without_infertility.xlsx")
data.head()

c:\Users\RESHAM\OneDrive\Desktop\programs\pcos_Copy\PCOS\app\ml


,Sl. No,Patient File No.,PCOS (Y/N),Age (yrs),Weight (Kg),Height(Cm),BMI,Blood Group,Pulse rate(bpm),RR (breaths/min),...,Fast food (Y/N),Reg.Exercise(Y/N),BP _Systolic (mmHg),BP _Diastolic (mmHg),Follicle No. (L),Follicle No. (R),Avg. F size (L) (mm),Avg. F size (R) (mm),Endometrium (mm),Unnamed: 44
0,11,11,0,20,71.0,163.0,NaN,15,80,20,...,0.0,0,110,80,7,15,17.0,20.0,6.0,NaN
1,28,28,0,20,68.0,152.0,NaN,17,72,20,...,0.0,0,110,80,3,2,10.0,11.0,10.0,NaN
2,40,40,0,20,74.0,171.0,NaN,13,74,16,...,0.0,0,110,70,6,12,13.0,12.0,0.0,NaN
3,240,240,0,20,56.0,159.0,NaN,13,72,18,...,0.0,1,120,80,10,5,15.0,18.0,8.0,NaN
4,298,298,0,20,52.0,150.0,NaN,15,72,18,...,1.0,0,110,70,7,9,13.0,17.0,9.6,NaN


In [3]:
data.columns = data.columns.str.strip()

data.drop([
    "Sl. No",
    "Patient File No.",
    "Blood Group",
    "BMI",
    "Pulse rate(bpm)",
    "RR (breaths/min)",
    "Hb(g/dl)",
    "Marraige Status (Yrs)",
    "Pregnant(Y/N)",
    "No. of aborptions",
    "I   beta-HCG(mIU/mL)",
    "II    beta-HCG(mIU/mL)",
    "FSH(mIU/mL)",
    "LH(mIU/mL)",
    "FSH/LH",
    "Hip(inch)",
    "Waist(inch)",
    "Waist:Hip Ratio",
    "TSH (mIU/L)",
    "AMH(ng/mL)",
    "PRL(ng/mL)",
    "Vit D3 (ng/mL)",
    "PRG(ng/mL)",
    "RBS(mg/dl)",
    "BP _Systolic (mmHg)",
    "BP _Diastolic (mmHg)",
    "Follicle No. (L)",
    "Follicle No. (R)",
    "Avg. F size (L) (mm)",
    "Avg. F size (R) (mm)",
    "Endometrium (mm)",
    "Unnamed: 44"
], axis=1, inplace=True, errors="ignore")  

print(data.columns)
data.to_csv("new_data.csv", index=False)

Index(['PCOS (Y/N)', 'Age (yrs)', 'Weight (Kg)', 'Height(Cm)', 'Cycle(R/I)',
       'Cycle length(days)', 'Weight gain(Y/N)', 'hair growth(Y/N)',
       'Skin darkening (Y/N)', 'Hair loss(Y/N)', 'Pimples(Y/N)',
       'Fast food (Y/N)', 'Reg.Exercise(Y/N)'],
      dtype='object')


In [4]:
data = data.apply(pd.to_numeric, errors='coerce')
data['Cycle(R/I)'] = data['Cycle(R/I)'].replace({2: 0, 4: 1})
data = data.fillna(data.mean(numeric_only=True))
data["BMI"] = data["Weight (Kg)"] / ((data["Height(Cm)"]/100) ** 2)

In [5]:
import numpy as np
binary_cols = [
    'Fast food (Y/N)',
    'Cycle(R/I)',
    'hair growth(Y/N)',
    'Skin darkening (Y/N)',
    'Hair loss(Y/N)',
    'Pimples(Y/N)',
    'Reg.Exercise(Y/N)'
]

for col in binary_cols:
    if data[col].mode().empty:
        data[col] = data[col].fillna(0)
    else:
        data[col] = data[col].fillna(data[col].mode()[0])

for col in data.columns:
    if data[col].isnull().sum() > 0:
        data[col] = data[col].fillna(data[col].median())

for col in data.columns:
    if data[col].isnull().sum() > 0:
        data[col] = data[col].fillna(data[col].median())

print("Total NaN left:", data.isnull().sum().sum())
print(data[['Cycle(R/I)', 'BMI']].head())

Total NaN left: 0
   Cycle(R/I)        BMI
0           0  26.722873
1           1  29.432133
2           1  25.306932
3           0  22.151023
4           0  23.111111


In [6]:
X = data[
    [
        'Age (yrs)',
        'Weight (Kg)',
        'Height(Cm)',
        'Fast food (Y/N)',
        'Cycle(R/I)',
        'hair growth(Y/N)',
        'Skin darkening (Y/N)',
        'Hair loss(Y/N)',
        'Pimples(Y/N)',
        'Reg.Exercise(Y/N)',
        'BMI'
    ]
]

y = data['PCOS (Y/N)']

print("Cleaned successfully")
print(X.head())


Cleaned successfully
   Age (yrs)  Weight (Kg)  Height(Cm)  Fast food (Y/N)  Cycle(R/I)  \
0         20         71.0       163.0              0.0           0   
1         20         68.0       152.0              0.0           1   
2         20         74.0       171.0              0.0           1   
3         20         56.0       159.0              0.0           0   
4         20         52.0       150.0              1.0           0   

   hair growth(Y/N)  Skin darkening (Y/N)  Hair loss(Y/N)  Pimples(Y/N)  \
0                 0                     0               0             0   
1                 0                     0               0             0   
2                 0                     0               0             0   
3                 0                     0               0             1   
4                 0                     0               1             1   

   Reg.Exercise(Y/N)        BMI  
0                  0  26.722873  
1                  0  29.432133  
2    

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# model = LogisticRegression(max_iter=1000)
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [13]:
import pickle
pickle.dump(model, open("models/pcos_model.pkl", "wb"))
pickle.dump(scaler, open("models/scaler.pkl", "wb"))

print("Model and Scaler Saved Successfully")

Model and Scaler Saved Successfully


In [17]:
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC-ROC:", roc_auc_score(y_test, y_prob))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.8343558282208589
AUC-ROC: 0.8854202401372213
F1 Score: 0.7610619469026548


In [18]:
import pandas as pd

sample = pd.DataFrame([{
    "Age (yrs)": 19,
    "Weight (Kg)": 63,
    "Height(Cm)": 145,
    "Fast food (Y/N)": 1,
    "Cycle(R/I)": 1, # 1->i rregular, 0->regular
    "hair growth(Y/N)": 0,
    "Skin darkening (Y/N)": 1,
    "Hair loss(Y/N)": 0,
    "Pimples(Y/N)": 0,
    "Reg.Exercise(Y/N)": 0,
    "BMI": 29.96
}])

sample_scaled = scaler.transform(sample)

prediction = model.predict(sample_scaled)[0]
probability = model.predict_proba(sample_scaled)[0][1]

if probability >= 0.65:
    risk = "High Risk"
elif probability >= 0.35:
    risk = "Moderate Risk"
else:
    risk = "Low Risk"
    
print("Prediction:", prediction)
print("Risk Probability:", round(probability * 100, 2), "%")
print("Risk Level:", risk)

Prediction: 1
Risk Probability: 90.87 %
Risk Level: High Risk
